# Model experiments (R-series)

Reads every run record under `models_artifacts/runs/` (written by
`python -m src.models.train --config configs/train.yaml [--final-test]`) and builds a
model-comparison table across runs -- cross-validation and, where present, final-test metrics.

`models_artifacts/` is gitignored (build output, regenerated per machine), so this notebook has
nothing to read until at least one training run has happened locally. Run
`python -m src.models.train --config configs/train.yaml --final-test` from `ml/` first.

In [1]:
import json
from pathlib import Path

import pandas as pd

ML_ROOT = Path.cwd() if (Path.cwd() / "models_artifacts").exists() else Path.cwd().parent
RUNS_DIR = ML_ROOT / "models_artifacts" / "runs"
REGISTRY_PATH = ML_ROOT / "src" / "models" / "model_registry" / "registry.json"

run_dirs = sorted(RUNS_DIR.glob("*")) if RUNS_DIR.exists() else []
print(f"Found {len(run_dirs)} run record(s) under {RUNS_DIR}")
for d in run_dirs:
    print(" -", d.name)

Found 1 run record(s) under C:\Users\RICHA\Desktop\Khetgpt\KhetGPT\ml\models_artifacts\runs
 - 20260927T174204Z


## Per-run, per-model cross-validation comparison

In [2]:
rows = []
for run_dir in run_dirs:
    metrics_path = run_dir / "metrics.json"
    if not metrics_path.exists():
        continue
    metrics = json.loads(metrics_path.read_text())
    for model_name, scores in metrics["cv"].items():
        rows.append({
            "run_id": run_dir.name,
            "model": model_name,
            "macro_f1_mean": scores["macro_f1"]["mean"],
            "macro_f1_std": scores["macro_f1"]["std"],
            "balanced_accuracy_mean": scores["balanced_accuracy"]["mean"],
            "balanced_accuracy_std": scores["balanced_accuracy"]["std"],
            "mcc_mean": scores["mcc"]["mean"],
            "accuracy_mean": scores["accuracy"]["mean"],
        })

cv_table = pd.DataFrame(rows).sort_values(["run_id", "macro_f1_mean"], ascending=[True, False])
cv_table

,run_id,model,macro_f1_mean,macro_f1_std,balanced_accuracy_mean,balanced_accuracy_std,mcc_mean,accuracy_mean
4,20260927T174204Z,random_forest,0.353921,0.027101,0.447321,0.044836,0.327741,0.550336
3,20260927T174204Z,xgboost,0.321957,0.044484,0.330638,0.049696,0.283068,0.563498
2,20260927T174204Z,logistic_regression,0.233906,0.007348,0.250268,0.003644,0.338980,0.598436
1,20260927T174204Z,stratified,0.166852,0.012678,0.169129,0.019314,0.019218,0.371144
0,20260927T174204Z,majority,0.107113,0.000119,0.166667,0.000000,0.000000,0.473489


## Final test-split metrics, where a run has them (`--final-test`)

In [3]:
test_rows = []
for run_dir in run_dirs:
    metrics_path = run_dir / "metrics.json"
    if not metrics_path.exists():
        continue
    metrics = json.loads(metrics_path.read_text())
    test = metrics.get("test")
    if not test:
        continue
    test_rows.append({
        "run_id": run_dir.name,
        "n_rows": test["n_rows"],
        "accuracy": test["accuracy"],
        "balanced_accuracy": test["balanced_accuracy"],
        "macro_f1": test["macro_f1"],
        "mcc": test["mcc"],
        "accuracy_ci95": test["bootstrap_ci_95"]["accuracy"],
        "macro_f1_ci95": test["bootstrap_ci_95"]["macro_f1"],
    })

test_table = pd.DataFrame(test_rows)
test_table

,run_id,n_rows,accuracy,balanced_accuracy,macro_f1,mcc,accuracy_ci95,macro_f1_ci95
0,20260927T174204Z,421,0.524941,0.435726,0.341088,0.301468,"[0.47743467933491684, 0.5748218527315915]","[0.26536676211800936, 0.41768366470669255]"


## Registered models (`registry.json`)

The registry (git-tracked, unlike `models_artifacts/`) is the source of truth for which model
version is actually deployed -- a run record above only means training happened locally, not
that it was registered/adopted.

In [4]:
if REGISTRY_PATH.exists():
    registry = json.loads(REGISTRY_PATH.read_text())
    reg_rows = [
        {
            "model_name": m["model_name"],
            "version": m["version"],
            "git_sha": m["git_sha"],
            "dataset_source": m["dataset_source"],
            "dataset_hash": m["dataset_hash"],
            "date": m["date"],
            "test_accuracy": (m.get("test_metrics") or {}).get("accuracy"),
        }
        for m in registry["models"]
    ]
    registry_table = pd.DataFrame(reg_rows)
else:
    registry_table = pd.DataFrame()
registry_table

,model_name,version,git_sha,dataset_source,dataset_hash,date,test_accuracy
0,fertilizer-classifier,0.1.0,f234d21,real,9eb97903,2026-09-27T13:24:19.172405+00:00,0.581948


## Per-crop breakdown and worst errors for the most recent run

Re-fits the registered pipeline config on the current train+val split and predicts on test, so
we get per-row predictions (not persisted by `train.py` itself) to slice by crop and inspect the
largest confusion patterns. See `docs/evaluation-report.md` §1.4-1.5 for the narrative version
of what shows up here.

In [5]:
import sys

sys.path.insert(0, str(ML_ROOT))

import yaml
from sklearn.preprocessing import LabelEncoder

from src.data_pipeline.feature_engineering import build_features
from src.evaluation.metrics import per_slice
from src.models.fertilizer_model import build_pipeline
from src.models.seeding import seed_everything
from src.models.train import _filtered, _records_from_frame, _sample_weight_kwargs, load_real_split

config = yaml.safe_load((ML_ROOT / "configs" / "train.yaml").read_text())
seed_everything(config["seed"])
excluded_classes = config["excluded_classes"]

df_train, target_train = load_real_split("train")
df_train, target_train, _ = _filtered(df_train, target_train, excluded_classes, "train")
real_val = load_real_split("val")
if real_val is not None:
    df_val, target_val = real_val
    df_val, target_val, _ = _filtered(df_val, target_val, excluded_classes, "val")
    df_all = pd.concat([df_train, df_val], ignore_index=True)
    target_all = pd.concat([target_train, target_val], ignore_index=True)
else:
    df_all, target_all = df_train, target_train

counts = target_all.value_counts()
unlearnable = counts[counts < 2].index.tolist()
if unlearnable:
    keep = ~target_all.isin(unlearnable)
    df_all, target_all = df_all[keep].reset_index(drop=True), target_all[keep].reset_index(drop=True)

X = build_features(_records_from_frame(df_all))
label_encoder = LabelEncoder().fit(target_all)
y_encoded = pd.Series(label_encoder.transform(target_all), index=target_all.index)

model_config = config["models"][config["candidate"]]
pipeline = build_pipeline(model_config["kind"], config["seed"], model_config)
pipeline.fit(X, y_encoded, **_sample_weight_kwargs(model_config, y_encoded))

df_test, target_test = load_real_split("test")
df_test, target_test, _ = _filtered(df_test, target_test, excluded_classes, "test")
X_test = build_features(_records_from_frame(df_test))
pred = label_encoder.inverse_transform(pipeline.predict(X_test))

crop_slice = per_slice(target_test.tolist(), pred.tolist(), df_test["crop_id"].tolist())
per_crop_table = pd.DataFrame(crop_slice).T.sort_values("accuracy", ascending=False)
per_crop_table

C:\Users\RICHA\Desktop\Khetgpt\KhetGPT\ml\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
C:\Users\RICHA\Desktop\Khetgpt\KhetGPT\ml\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
C:\Users\RICHA\Desktop\Khetgpt\KhetGPT\ml\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


,accuracy,balanced_accuracy,macro_f1,n
rice,0.762712,0.400000,0.372549,59.0
sugarcane,0.706897,0.283333,0.266667,58.0
cotton,0.661538,0.250000,0.160748,65.0
maize,0.460317,0.392361,0.374661,63.0
chickpea,0.373134,0.333333,0.181159,67.0
wheat,0.355932,0.205882,0.140000,59.0
barley,0.340000,0.182143,0.139143,50.0


In [6]:
results_df = df_test.copy()
results_df["y_true"] = target_test.values
results_df["y_pred"] = pred
wrong = results_df[results_df["y_true"] != results_df["y_pred"]]
print(f"Misclassified: {len(wrong)} / {len(results_df)}")
wrong.groupby(["crop_id", "y_true", "y_pred"]).size().sort_values(ascending=False).head(15)

Misclassified: 200 / 421


crop_id    y_true      y_pred      
chickpea   np_28_28_0  dap             35
cotton     np_28_28_0  urea            15
maize      urea        np_28_28_0      14
wheat      urea        np_28_28_0      13
           np_28_28_0  np_20_20_0      13
rice       np_28_28_0  urea            12
maize      np_28_28_0  urea            11
barley     urea        np_28_28_0      10
sugarcane  urea        npk_17_17_17    10
barley     np_28_28_0  np_20_20_0       9
wheat      np_20_20_0  np_28_28_0       7
chickpea   np_20_20_0  dap              7
barley     urea        np_20_20_0       7
cotton     np_20_20_0  urea             6
barley     np_20_20_0  np_28_28_0       5
dtype: int64